# 2-D Tubular Reactor Flow (Cylindrical Coordinates - Steady State)

## ChBE 3300: Multidimensional Fluids and Heat Transport

### Problem Description

This notebook simulates **steady-state laminar flow and chemical reaction** in a tubular packed-bed reactor, a fundamental unit operation in chemical engineering used for:
- Catalytic reactions (e.g., ammonia synthesis, Fischer-Tropsch)
- Heterogeneous catalysis
- Gas-phase reactions
- Petroleum refining processes

### Physical System

A cylindrical tubular reactor where:
- **Flow**: Axisymmetric laminar flow (Hagen-Poiseuille)
- **Reaction**: First-order catalytic reaction A → B at the wall
- **Geometry**: Cylindrical tube with radius R and length L
- **Coordinates**: Cylindrical (r, z) with axial symmetry (no θ dependence)

### Governing Equations (Cylindrical Coordinates - Steady State)

#### 1. Continuity Equation (Incompressible, Axisymmetric)
$$\frac{1}{r}\frac{\partial(rv_r)}{\partial r} + \frac{\partial v_z}{\partial z} = 0$$

#### 2. Navier-Stokes Equations (Steady State, Axisymmetric)

**r-momentum:**
$$\rho\left(v_r\frac{\partial v_r}{\partial r} + v_z\frac{\partial v_r}{\partial z}\right) = -\frac{\partial p}{\partial r} + \mu\left[\frac{1}{r}\frac{\partial}{\partial r}\left(r\frac{\partial v_r}{\partial r}\right) - \frac{v_r}{r^2} + \frac{\partial^2 v_r}{\partial z^2}\right]$$

**z-momentum:**
$$\rho\left(v_r\frac{\partial v_z}{\partial r} + v_z\frac{\partial v_z}{\partial z}\right) = -\frac{\partial p}{\partial z} + \mu\left[\frac{1}{r}\frac{\partial}{\partial r}\left(r\frac{\partial v_z}{\partial r}\right) + \frac{\partial^2 v_z}{\partial z^2}\right]$$

#### 3. Steady-State Convection-Diffusion-Reaction Equation
$$v_r\frac{\partial C_A}{\partial r} + v_z\frac{\partial C_A}{\partial z} = D_{AB}\left[\frac{1}{r}\frac{\partial}{\partial r}\left(r\frac{\partial C_A}{\partial r}\right) + \frac{\partial^2 C_A}{\partial z^2}\right] - k C_A$$

where:
- $v_r, v_z$ = velocity components in r and z directions (m/s)
- $p$ = pressure (Pa)
- $C_A$ = concentration of reactant A (mol/m³)
- $D_{AB}$ = molecular diffusion coefficient (m²/s)
- $k$ = reaction rate constant (1/s)

### Analytical Solution for Fully Developed Flow

For fully developed laminar flow in a circular tube (Hagen-Poiseuille):

$$v_z(r) = v_{z,max}\left(1 - \frac{r^2}{R^2}\right) = 2v_{avg}\left(1 - \frac{r^2}{R^2}\right)$$

where $v_{z,max} = 2v_{avg}$ and $v_r = 0$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Set plotting style
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

## 1. Physical Parameters and Reactor Geometry

In [ ]:
# Reactor geometry
R = 0.05              # Tube radius (5 cm)
L = 2.0               # Tube length (2 m)

# Fluid properties (typical gas at moderate conditions)
rho = 1.2             # Density (kg/m³) - gas
mu = 1.8e-5           # Dynamic viscosity (Pa·s) - air-like
nu = mu / rho         # Kinematic viscosity (m²/s)

# Flow parameters
v_avg = 0.5           # Average axial velocity (m/s)
v_max = 2 * v_avg     # Maximum velocity (centerline)
Q = np.pi * R**2 * v_avg  # Volumetric flow rate (m³/s)
Re = rho * v_avg * (2*R) / mu  # Reynolds number (based on diameter)

# Mass transfer and reaction parameters
D_AB = 2e-5           # Molecular diffusivity (m²/s) - gas-phase
k_rxn = 0.1           # First-order reaction rate constant (1/s)
C_A0 = 10.0           # Inlet concentration (mol/m³)

# Dimensionless numbers
Pe = v_avg * L / D_AB              # Péclet number
Da = k_rxn * L / v_avg             # Damköhler number
Sc = nu / D_AB                     # Schmidt number
tau = L / v_avg                    # Residence time

print("=" * 70)
print("TUBULAR REACTOR SIMULATION - CYLINDRICAL COORDINATES (STEADY STATE)")
print("=" * 70)
print(f"\nReactor Geometry:")
print(f"  Tube radius (R): {R*100:.1f} cm")
print(f"  Tube length (L): {L:.2f} m")
print(f"  L/D ratio: {L/(2*R):.1f}")
print(f"\nFluid Properties:")
print(f"  Density: {rho:.2f} kg/m³")
print(f"  Viscosity: {mu*1e6:.2f} μPa·s")
print(f"  Diffusivity: {D_AB:.2e} m²/s")
print(f"\nFlow Conditions:")
print(f"  Average velocity: {v_avg:.2f} m/s")
print(f"  Maximum velocity: {v_max:.2f} m/s")
print(f"  Volumetric flow rate: {Q*1e6:.2f} L/min")
print(f"  Residence time: {tau:.2f} s")
print(f"\nReaction Parameters:")
print(f"  Inlet concentration: {C_A0:.2f} mol/m³")
print(f"  Reaction rate constant: {k_rxn:.3f} 1/s")
print(f"\nDimensionless Groups:")
print(f"  Reynolds number (Re): {Re:.1f}")
print(f"  Péclet number (Pe): {Pe:.1f}")
print(f"  Damköhler number (Da): {Da:.3f}")
print(f"  Schmidt number (Sc): {Sc:.2f}")
print(f"\nFlow Regime: {'Laminar' if Re < 2300 else 'Turbulent'}")
print(f"Reaction Regime: {'Kinetically controlled' if Da < 0.1 else 'Diffusion influenced' if Da < 10 else 'Mass transfer controlled'}")
print("=" * 70)

## 2. Numerical Grid Setup

In [ ]:
# Grid parameters
nr = 50               # Number of grid points in radial direction
nz = 100              # Number of grid points in axial direction

# Create grid
r = np.linspace(0, R, nr)        # Radial coordinate (0 to R)
z = np.linspace(0, L, nz)        # Axial coordinate (0 to L)
dr = r[1] - r[0]
dz = z[1] - z[0]

# Create meshgrid
Z, R_grid = np.meshgrid(z, r)

print(f"\nGrid Setup:")
print(f"  Radial points (nr): {nr}")
print(f"  Axial points (nz): {nz}")
print(f"  Grid spacing: Δr = {dr*1000:.2f} mm, Δz = {dz*100:.2f} cm")
print(f"  Total grid points: {nr * nz}")

## 3. Velocity Field - Hagen-Poiseuille Flow

For fully developed laminar flow in a circular tube:
- Axial velocity has parabolic profile
- Radial velocity is zero
- Flow is independent of z (fully developed)

In [ ]:
# Initialize velocity fields
v_r = np.zeros((nr, nz))         # Radial velocity (zero for fully developed flow)
v_z = np.zeros((nr, nz))         # Axial velocity

# Calculate parabolic velocity profile (Hagen-Poiseuille)
for i in range(nr):
    v_z[i, :] = v_max * (1 - (r[i]/R)**2)

# Visualize velocity field
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Velocity profile
ax1.plot(v_z[:, 0], r*100, 'b-', linewidth=3)
ax1.axhline(0, color='k', linewidth=2, label='Centerline')
ax1.axhline(R*100, color='k', linewidth=2, label='Wall')
ax1.axvline(v_avg, color='r', linestyle='--', linewidth=2, label=f'Average = {v_avg:.2f} m/s')
ax1.fill_betweenx(r*100, 0, v_z[:, 0], alpha=0.3)
ax1.set_xlabel('Axial Velocity, $v_z$ (m/s)', fontsize=12)
ax1.set_ylabel('Radial Position, r (cm)', fontsize=12)
ax1.set_title('Hagen-Poiseuille Velocity Profile', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, v_max*1.1)

# 2D velocity field
contour = ax2.contourf(Z, R_grid*100, v_z, levels=20, cmap='RdYlBu_r')
plt.colorbar(contour, ax=ax2, label='Axial Velocity (m/s)')
ax2.set_xlabel('Axial Position, z (m)', fontsize=12)
ax2.set_ylabel('Radial Position, r (cm)', fontsize=12)
ax2.set_title('Velocity Field in Reactor', fontsize=13, fontweight='bold')
ax2.set_ylim(0, R*100)

plt.tight_layout()
plt.show()

print(f"\nVelocity Field Summary:")
print(f"  Centerline velocity: {v_z[0, 0]:.3f} m/s")
print(f"  Wall velocity: {v_z[-1, 0]:.3f} m/s")
print(f"  Average velocity: {np.mean(v_z[:, 0]):.3f} m/s")
print(f"  Velocity ratio (max/avg): {v_max/v_avg:.2f}")

## 4. Solve Steady-State Convection-Diffusion-Reaction Equation

### Finite Difference Discretization (Cylindrical Coordinates)

For steady-state with axisymmetry:

$$v_z\frac{\partial C_A}{\partial z} = D_{AB}\left[\frac{1}{r}\frac{\partial}{\partial r}\left(r\frac{\partial C_A}{\partial r}\right) + \frac{\partial^2 C_A}{\partial z^2}\right] - k C_A$$

Using central differences and solving iteratively:

In [ ]:
# Initialize concentration field
C_A = np.zeros((nr, nz))
C_A[:, 0] = C_A0  # Inlet boundary condition

# Iterative solver parameters
max_iter = 10000
tolerance = 1e-6
omega = 1.5  # Relaxation factor for SOR (Successive Over-Relaxation)

print("\nSolving steady-state convection-diffusion-reaction equation...")
print("Method: Successive Over-Relaxation (SOR)")
print("Progress: ", end="")

# Iterative solution
for iteration in range(max_iter):
    C_A_old = C_A.copy()
    
    # Update interior points
    for i in range(1, nr-1):
        for j in range(1, nz-1):
            # Radial diffusion term (with 1/r factor for cylindrical coordinates)
            if i == 0:  # Centerline - use L'Hôpital's rule
                radial_diff = 2 * D_AB * (C_A[i+1, j] - C_A[i, j]) / dr**2
            else:
                radial_diff = D_AB * (
                    (C_A[i+1, j] - C_A[i, j]) / dr * (r[i] + dr/2) / r[i] / dr +
                    (C_A[i, j] - C_A[i-1, j]) / dr * (r[i] - dr/2) / r[i] / dr
                )
            
            # Axial diffusion term
            axial_diff = D_AB * (C_A[i, j+1] - 2*C_A[i, j] + C_A[i, j-1]) / dz**2
            
            # Convection term (upwind scheme for stability)
            convection = v_z[i, j] * (C_A[i, j] - C_A[i, j-1]) / dz
            
            # Reaction term
            reaction = -k_rxn * C_A[i, j]
            
            # Update with relaxation
            residual = (radial_diff + axial_diff - convection + reaction) * dz / v_z[i, j]
            C_A[i, j] = C_A[i, j] + omega * residual
    
    # Boundary conditions
    C_A[:, 0] = C_A0              # Inlet
    C_A[0, :] = C_A[1, :]         # Centerline (symmetry)
    C_A[-1, :] = 0.0              # Wall (catalytic reaction - zero concentration)
    C_A[:, -1] = C_A[:, -2]       # Outlet (zero gradient)
    
    # Prevent negative concentrations
    C_A = np.maximum(C_A, 0)
    
    # Check convergence
    error = np.max(np.abs(C_A - C_A_old))
    
    if iteration % 1000 == 0:
        print(f"{iteration} ", end="", flush=True)
    
    if error < tolerance:
        print(f"\n\nConverged after {iteration} iterations")
        print(f"Final error: {error:.2e}")
        break
else:
    print(f"\n\nWarning: Maximum iterations reached. Error: {error:.2e}")

# Calculate conversion
C_outlet_avg = np.trapz(C_A[:, -1] * v_z[:, -1] * 2 * np.pi * r, r) / Q
conversion = (C_A0 - C_outlet_avg) / C_A0 * 100

print(f"\nResults:")
print(f"  Inlet concentration: {C_A0:.3f} mol/m³")
print(f"  Outlet avg concentration: {C_outlet_avg:.3f} mol/m³")
print(f"  Conversion: {conversion:.2f}%")

## 5. Visualization of Concentration Field

In [ ]:
# Create comprehensive visualization
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# 1. Concentration field (2D)
ax1 = fig.add_subplot(gs[0, :])
contour = ax1.contourf(Z, R_grid*100, C_A, levels=30, cmap='viridis')
contour_lines = ax1.contour(Z, R_grid*100, C_A, levels=10, colors='white', 
                             alpha=0.4, linewidths=0.8)
ax1.clabel(contour_lines, inline=True, fontsize=8, fmt='%.1f')
cbar = plt.colorbar(contour, ax=ax1, label='Concentration $C_A$ (mol/m³)')
ax1.set_xlabel('Axial Position, z (m)', fontsize=12)
ax1.set_ylabel('Radial Position, r (cm)', fontsize=12)
ax1.set_title('Steady-State Concentration Field in Tubular Reactor', 
              fontsize=14, fontweight='bold')
ax1.set_ylim(0, R*100)

# 2. Radial profiles at different axial positions
ax2 = fig.add_subplot(gs[1, 0])
z_positions = [0, nz//4, nz//2, 3*nz//4, nz-1]
z_labels = ['Inlet', 'L/4', 'L/2', '3L/4', 'Outlet']
colors = ['blue', 'green', 'orange', 'red', 'purple']

for z_idx, z_label, color in zip(z_positions, z_labels, colors):
    ax2.plot(C_A[:, z_idx], r*100, linewidth=2.5, label=z_label, color=color)

ax2.set_xlabel('Concentration $C_A$ (mol/m³)', fontsize=11)
ax2.set_ylabel('Radial Position, r (cm)', fontsize=11)
ax2.set_title('Radial Concentration Profiles', fontsize=12, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, R*100)

# 3. Axial profiles at different radial positions
ax3 = fig.add_subplot(gs[1, 1])
r_positions = [0, nr//4, nr//2, 3*nr//4]
r_labels = ['Centerline', 'r/R=0.25', 'r/R=0.5', 'r/R=0.75']
colors2 = ['blue', 'green', 'orange', 'red']

for r_idx, r_label, color in zip(r_positions, r_labels, colors2):
    ax3.plot(z, C_A[r_idx, :], linewidth=2.5, label=r_label, color=color)

ax3.set_xlabel('Axial Position, z (m)', fontsize=11)
ax3.set_ylabel('Concentration $C_A$ (mol/m³)', fontsize=11)
ax3.set_title('Axial Concentration Profiles', fontsize=12, fontweight='bold')
ax3.legend(loc='best', fontsize=10)
ax3.grid(True, alpha=0.3)

# 4. Cup-mixing (bulk) concentration along reactor
ax4 = fig.add_subplot(gs[2, 0])
C_bulk = np.zeros(nz)
for j in range(nz):
    # Cup-mixing concentration: integral of C*v*r*dr / integral of v*r*dr
    C_bulk[j] = np.trapz(C_A[:, j] * v_z[:, j] * 2 * np.pi * r, r) / Q

# Theoretical plug flow with reaction
C_plug = C_A0 * np.exp(-k_rxn * z / v_avg)

ax4.plot(z, C_bulk, 'b-', linewidth=3, label='CFD Solution (with diffusion)')
ax4.plot(z, C_plug, 'r--', linewidth=2, label='Plug Flow (no diffusion)')
ax4.fill_between(z, 0, C_bulk, alpha=0.3)
ax4.set_xlabel('Axial Position, z (m)', fontsize=11)
ax4.set_ylabel('Bulk Concentration $\overline{C_A}$ (mol/m³)', fontsize=11)
ax4.set_title('Bulk Average Concentration vs Plug Flow', fontsize=12, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)

# 5. Conversion along reactor
ax5 = fig.add_subplot(gs[2, 1])
X_local = (C_A0 - C_bulk) / C_A0 * 100
X_plug = (C_A0 - C_plug) / C_A0 * 100

ax5.plot(z, X_local, 'b-', linewidth=3, label='CFD Solution')
ax5.plot(z, X_plug, 'r--', linewidth=2, label='Plug Flow')
ax5.fill_between(z, 0, X_local, alpha=0.3)
ax5.set_xlabel('Axial Position, z (m)', fontsize=11)
ax5.set_ylabel('Conversion X (%)', fontsize=11)
ax5.set_title('Conversion Along Reactor Length', fontsize=12, fontweight='bold')
ax5.legend(fontsize=10)
ax5.grid(True, alpha=0.3)
ax5.set_ylim(0, 100)

plt.suptitle('Tubular Reactor Analysis - Cylindrical Coordinates', 
             fontsize=15, fontweight='bold', y=0.995)
plt.show()

## 6. 3D Visualization (Axisymmetric Representation)

In [ ]:
# Create 3D representation of axisymmetric problem
fig = plt.figure(figsize=(16, 6))

# Subsample for clearer visualization
skip_r = 2
skip_z = 3
theta = np.linspace(0, 2*np.pi, 30)

# Create cylindrical coordinates for 3D plot
r_3d = r[::skip_r]
z_3d = z[::skip_z]
C_3d = C_A[::skip_r, ::skip_z]

# Left plot: Full 3D cylinder
ax1 = fig.add_subplot(121, projection='3d')
for i, r_val in enumerate(r_3d[::2]):
    X_cyl = r_val * np.cos(theta)
    Y_cyl = r_val * np.sin(theta)
    for j, z_val in enumerate(z_3d[::3]):
        Z_cyl = z_val * np.ones_like(theta)
        C_val = C_3d[i*2, j*3]
        color = plt.cm.viridis(C_val / C_A0)
        ax1.plot(Z_cyl, X_cyl, Y_cyl, color=color, linewidth=1, alpha=0.6)

ax1.set_xlabel('Axial Position z (m)', fontsize=10)
ax1.set_ylabel('x (m)', fontsize=10)
ax1.set_zlabel('y (m)', fontsize=10)
ax1.set_title('3D Reactor Geometry\n(Concentration Contours)', fontsize=12, fontweight='bold')

# Right plot: Cross-section with concentration
ax2 = fig.add_subplot(122, projection='3d')
Z_mesh, R_mesh = np.meshgrid(z[::skip_z], r[::skip_r])
X_mesh = R_mesh * np.cos(0)  # Half-section at theta=0
Y_mesh = R_mesh * np.sin(0)

surf = ax2.plot_surface(Z_mesh, X_mesh, C_A[::skip_r, ::skip_z], 
                        cmap='viridis', alpha=0.9, edgecolor='none')
# Mirror image
ax2.plot_surface(Z_mesh, -X_mesh, C_A[::skip_r, ::skip_z], 
                cmap='viridis', alpha=0.9, edgecolor='none')

ax2.set_xlabel('Axial Position z (m)', fontsize=10)
ax2.set_ylabel('Radial Position (m)', fontsize=10)
ax2.set_zlabel('Concentration (mol/m³)', fontsize=10)
ax2.set_title('Concentration Surface\n(Axisymmetric)', fontsize=12, fontweight='bold')
fig.colorbar(surf, ax=ax2, label='$C_A$ (mol/m³)', shrink=0.6)

plt.tight_layout()
plt.show()

## 7. Reactor Performance Analysis

In [ ]:
# Calculate key performance metrics

# 1. Overall conversion
X_overall = conversion

# 2. Theoretical conversions for comparison
X_PFR = (1 - np.exp(-k_rxn * tau)) * 100  # Plug Flow Reactor
X_CSTR = (k_rxn * tau / (1 + k_rxn * tau)) * 100  # CSTR

# 3. Reactor volume and productivity
V_reactor = np.pi * R**2 * L  # m³
n_A_in = Q * C_A0  # Molar flow rate in (mol/s)
n_A_out = Q * C_outlet_avg  # Molar flow rate out (mol/s)
n_A_reacted = n_A_in - n_A_out  # Molar rate of reaction (mol/s)

# 4. Space-time and space-velocity
space_time = V_reactor / Q  # s
space_velocity = Q / V_reactor  # 1/s

# 5. Mass transfer coefficient at wall (Sherwood number)
# Sh = k_m * D / D_AB for mass transfer
concentration_gradient_wall = (C_A[-2, nz//2] - C_A[-1, nz//2]) / dr
flux_wall = -D_AB * concentration_gradient_wall

print("\n" + "="*70)
print("REACTOR PERFORMANCE SUMMARY")
print("="*70)
print(f"\nReactor Design:")
print(f"  Volume: {V_reactor*1000:.2f} L")
print(f"  Space-time: {space_time:.2f} s")
print(f"  Space-velocity: {space_velocity:.4f} s⁻¹")
print(f"\nFlow Rates:")
print(f"  Volumetric flow: {Q*60*1000:.2f} L/min")
print(f"  Molar feed rate: {n_A_in*1000:.3f} mmol/s")
print(f"  Molar exit rate: {n_A_out*1000:.3f} mmol/s")
print(f"  Reaction rate: {n_A_reacted*1000:.3f} mmol/s")
print(f"\nConversion:")
print(f"  CFD model (with diffusion): {X_overall:.2f}%")
print(f"  Ideal PFR (plug flow): {X_PFR:.2f}%")
print(f"  Ideal CSTR: {X_CSTR:.2f}%")
print(f"\nReactor Efficiency:")
print(f"  vs. PFR: {(X_overall/X_PFR)*100:.1f}%")
print(f"  vs. CSTR: {(X_overall/X_CSTR)*100:.1f}%")
print(f"\nMass Transfer:")
print(f"  Wall flux (mid-reactor): {flux_wall*1e6:.3f} μmol/(m²·s)")
print(f"  Average wall concentration: {np.mean(C_A[-1, :]):.3f} mol/m³")
print("="*70)

# Create comparison bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Conversion comparison
reactor_types = ['CFD Model\n(Dispersion)', 'Plug Flow\nReactor', 'CSTR']
conversions = [X_overall, X_PFR, X_CSTR]
colors_bar = ['#2E86AB', '#A23B72', '#F18F01']

bars = ax1.bar(reactor_types, conversions, color=colors_bar, edgecolor='black', linewidth=2)
ax1.set_ylabel('Conversion (%)', fontsize=12)
ax1.set_title('Reactor Performance Comparison', fontsize=13, fontweight='bold')
ax1.set_ylim(0, max(conversions)*1.2)
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, conv in zip(bars, conversions):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{conv:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Concentration profile comparison
ax2.plot(z, C_bulk, 'b-', linewidth=3, label='CFD (Parabolic flow)')
ax2.plot(z, C_plug, 'r--', linewidth=2.5, label='PFR (Plug flow)')
ax2.plot(z, C_A0 / (1 + k_rxn * z / v_avg), 'g:', linewidth=2.5, label='CSTR cascade')
ax2.set_xlabel('Axial Position, z (m)', fontsize=12)
ax2.set_ylabel('Bulk Concentration (mol/m³)', fontsize=12)
ax2.set_title('Concentration Decay Comparison', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10, loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Summary and Chemical Engineering Applications

### Key Findings:

1. **Flow Pattern**: 
   - Parabolic velocity profile (Hagen-Poiseuille)
   - Maximum velocity at centerline = 2 × average velocity
   - Laminar flow regime (Re << 2300)

2. **Concentration Distribution**:
   - Radial gradients due to wall reaction and parabolic flow
   - Lower conversion than ideal PFR due to velocity distribution
   - Dispersion effects reduce reactor efficiency

3. **Reactor Performance**:
   - Conversion depends on Damköhler number (Da)
   - Wall catalysis creates concentration boundary layer
   - Radial diffusion important for mass transfer to catalyst

### Industrial Applications:

#### 1. **Catalytic Reactors**
   - Steam reforming: CH₄ + H₂O → CO + 3H₂
   - Ammonia synthesis: N₂ + 3H₂ → 2NH₃
   - Fischer-Tropsch: CO + H₂ → hydrocarbons

#### 2. **Petroleum Refining**
   - Catalytic cracking
   - Hydrotreating
   - Reforming processes

#### 3. **Chemical Production**
   - Oxidation reactions (ethylene oxide, maleic anhydride)
   - Polymerization reactors
   - Selective oxidation

#### 4. **Environmental Engineering**
   - Catalytic converters (automotive)
   - NOx reduction (SCR)
   - VOC oxidation

### Design Considerations:

1. **Maximize Conversion**:
   - Increase residence time (longer reactor or lower flow rate)
   - Increase temperature (higher k)
   - Use multiple reactors in series

2. **Improve Mass Transfer**:
   - Reduce tube diameter (shorter diffusion path)
   - Increase turbulence (higher Re)
   - Use porous catalysts (increase surface area)

3. **Optimize Economics**:
   - Trade-off between conversion and pressure drop
   - Balance capital cost vs. operating cost
   - Consider heat management (exothermic/endothermic)

## 9. Exercises

**Exercise 1**: Investigate the effect of Reynolds number on the concentration field. Increase the flow velocity and observe changes.

**Exercise 2**: Modify the boundary condition at the wall to include mass transfer resistance: $-D\frac{\partial C}{\partial r}\bigg|_{r=R} = k_m(C_s - C_w)$

**Exercise 3**: Implement a second-order reaction: $r = k C_A^2$. How does this affect the concentration profile?

**Exercise 4**: Add a reversible reaction: $A \rightleftharpoons B$. Study the approach to equilibrium.

**Exercise 5**: Calculate the Sherwood number along the reactor length and compare with correlations from literature.

**Exercise 6**: Optimize the reactor length for 95% conversion. What is the minimum L/D ratio needed?

In [ ]:
# Your code here for exercises


## References

1. Fogler, H. S. (2016). *Elements of Chemical Reaction Engineering* (5th ed.). Prentice Hall.

2. Bird, R. B., Stewart, W. E., & Lightfoot, E. N. (2007). *Transport Phenomena* (2nd ed.). John Wiley & Sons.

3. Levenspiel, O. (1999). *Chemical Reaction Engineering* (3rd ed.). John Wiley & Sons.

4. Aris, R. (1975). *The Mathematical Theory of Diffusion and Reaction in Permeable Catalysts*. Oxford University Press.

5. Froment, G. F., Bischoff, K. B., & De Wilde, J. (2010). *Chemical Reactor Analysis and Design* (3rd ed.). Wiley.